# KG-Bayesian-Prior Demo

This notebook demonstrates the key features of the **Relation-Aware GP Prior for Entity-Level Uncertainty Quantification in Knowledge Graphs**.

## Contents
1. Data Loading
2. Baseline Models
3. GP-KGE Model
4. Uncertainty Analysis
5. Visualization

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_fb15k237
from src.models import TransE, DistMult, ComplEx, GPKGE
from src.utils.training import set_seed

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Data Loading

In [ ]:
# Load dataset (will create sample data if real data not available)
train_data, valid_data, test_data = load_fb15k237()

print(f'Entities: {train_data.num_entities}')
print(f'Relations: {train_data.num_relations}')
print(f'Train triples: {len(train_data)}')
print(f'Valid triples: {len(valid_data)}')
print(f'Test triples: {len(test_data)}')

In [ ]:
# Visualize degree distribution
degrees = train_data.get_entity_degrees()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(degrees, bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('Degree')
ax.set_ylabel('Count')
ax.set_title('Entity Degree Distribution')
ax.set_yscale('log')
plt.show()

print(f'Mean degree: {degrees.mean():.2f}')
print(f'Max degree: {degrees.max()}')
print(f'Isolated entities: {(degrees == 0).sum()}')

## 2. Baseline Models

In [ ]:
# Create baseline models
embedding_dim = 50

models = {
    'TransE': TransE(train_data.num_entities, train_data.num_relations, embedding_dim),
    'DistMult': DistMult(train_data.num_entities, train_data.num_relations, embedding_dim),
    'ComplEx': ComplEx(train_data.num_entities, train_data.num_relations, embedding_dim),
}

for name, model in models.items():
    num_params = sum(p.numel() for p in model.parameters())
    print(f'{name}: {num_params:,} parameters')

## 3. GP-KGE Model (Our Contribution)

In [ ]:
# Create GP-KGE model
gp_model = GPKGE(
    num_entities=train_data.num_entities,
    num_relations=train_data.num_relations,
    embedding_dim=embedding_dim,
    kernel_type='relation_aware',
    scoring_function='distmult',
    num_inducing=min(100, train_data.num_entities),
)

# Set graph structure
gp_model.set_graph(train_data)

num_params = sum(p.numel() for p in gp_model.parameters())
print(f'GP-KGE: {num_params:,} parameters')

In [ ]:
# Examine kernel parameters
kernel = gp_model.kernel

print('Relation-Aware Kernel Parameters:')
print(f'  Lengthscales: {kernel.lengthscale.detach().numpy()}')
print(f'  Variances: {kernel.variance.detach().numpy()}')

## 4. Uncertainty Analysis

In [ ]:
# Get entity-level uncertainty from GP-KGE
with torch.no_grad():
    all_entities = torch.arange(train_data.num_entities)
    uncertainties = gp_model.get_entity_uncertainty(all_entities).numpy()

print(f'Uncertainty range: [{uncertainties.min():.4f}, {uncertainties.max():.4f}]')
print(f'Mean uncertainty: {uncertainties.mean():.4f}')

In [ ]:
# Visualize uncertainty vs degree
fig, ax = plt.subplots(figsize=(10, 5))

ax.scatter(degrees, uncertainties, alpha=0.5, s=10)
ax.set_xlabel('Entity Degree')
ax.set_ylabel('Uncertainty')
ax.set_title('Entity Uncertainty vs Connectivity')
ax.set_xscale('log')

# Add trend line
z = np.polyfit(np.log(degrees + 1), uncertainties, 1)
p = np.poly1d(z)
x_line = np.logspace(0, np.log10(degrees.max() + 1), 100)
ax.plot(x_line, p(np.log(x_line)), 'r--', label='Trend')
ax.legend()

plt.tight_layout()
plt.show()

# Correlation
from scipy.stats import spearmanr
corr, p_val = spearmanr(degrees, uncertainties)
print(f'Spearman correlation: {corr:.4f} (p={p_val:.4e})')

In [ ]:
# Uncertainty for specific triples
sample_triples = test_data.triples[:10]

h = torch.tensor(sample_triples[:, 0])
r = torch.tensor(sample_triples[:, 1])
t = torch.tensor(sample_triples[:, 2])

with torch.no_grad():
    pred = gp_model.predict_with_uncertainty(h, r, t)

print('Triple Predictions with Uncertainty:')
print(f'{"Score":<10} {"Epistemic":<12} {"Total":<10}')
print('-' * 35)
for i in range(len(sample_triples)):
    print(f'{pred["mean"][i].item():<10.4f} '
          f'{pred["epistemic"][i].item():<12.4f} '
          f'{pred["total"][i].item():<10.4f}')

## 5. Visualization

In [ ]:
# Visualize learned relation importance
if hasattr(kernel, 'get_relation_importance'):
    importance = kernel.get_relation_importance().detach().numpy()
    
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(range(len(importance)), importance)
    ax.set_xlabel('Relation ID')
    ax.set_ylabel('Importance')
    ax.set_title('Learned Relation Importance')
    plt.tight_layout()
    plt.show()

In [ ]:
# t-SNE visualization of entity embeddings with uncertainty
from sklearn.manifold import TSNE

# Get embeddings
with torch.no_grad():
    embeddings = gp_model.entity_mean.numpy()

# Subsample for visualization
n_sample = min(1000, len(embeddings))
indices = np.random.choice(len(embeddings), n_sample, replace=False)

# Run t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
emb_2d = tsne.fit_transform(embeddings[indices])

# Plot with uncertainty as color
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    emb_2d[:, 0], emb_2d[:, 1],
    c=uncertainties[indices],
    cmap='viridis',
    alpha=0.6,
    s=10
)
plt.colorbar(scatter, label='Uncertainty')
ax.set_title('Entity Embeddings (color = uncertainty)')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

## Summary

This demo showed:
1. **Data loading** for KG benchmarks
2. **Baseline models**: TransE, DistMult, ComplEx
3. **GP-KGE model** with relation-aware kernel
4. **Uncertainty quantification** at entity and triple level
5. **Visualization** of embeddings and uncertainties

Key findings:
- Uncertainty is inversely correlated with entity degree (as expected)
- Different relations contribute differently to the kernel
- The model provides both epistemic and aleatoric uncertainty estimates